In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import re

warnings.filterwarnings("ignore")

In [2]:
BASE_DIR = Path(r"C:\Users\Shriyans S Sahoo\IIIT Dharwad\Academics\Sem 4\Time Series Forecasting\TimeSeries Project")

XLS_DIR = BASE_DIR / "nldc_xls"

YEAR_FOLDERS = [
    XLS_DIR / "2023-24",
    XLS_DIR / "2024-25",
    XLS_DIR / "2025-26"
]

# Output folders
OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR)

Output folder: C:\Users\Shriyans S Sahoo\IIIT Dharwad\Academics\Sem 4\Time Series Forecasting\TimeSeries Project\data\processed


In [3]:
def extract_date_from_filename(file_path):
    try:
        return pd.to_datetime(file_path.stem.split("_")[0])
    except:
        return None

## Part A: Daily Data (MOP_E)

In [4]:
def extract_daily_peak(file_path):
    try:
        # Read sheet without assuming structure
        df = pd.read_excel(file_path, sheet_name="MOP_E", header=None)

        for i in range(len(df)):
            row = df.iloc[i]

            # Convert row to string for search
            row_text = " ".join(row.astype(str)).lower()

            if "maximum demand met during the day" in row_text:

                # Extract numeric values only
                numeric_values = []

                for val in row:
                    try:
                        num = float(val)
                        numeric_values.append(num)
                    except:
                        continue

                if len(numeric_values) == 0:
                    return None

                # Last numeric value = TOTAL
                return numeric_values[-1]

        return None

    except:
        return None

In [5]:
daily_data = []

for folder in YEAR_FOLDERS:
    print(f"\nProcessing {folder}")

    for file_path in folder.rglob("*.xls*"):

        date = extract_date_from_filename(file_path)
        if date is None:
            continue

        peak = extract_daily_peak(file_path)

        if peak is not None:
            daily_data.append({
                "date": date,
                "max_demand_met_mw": peak
            })

daily_df = pd.DataFrame(daily_data)

print("Daily dataset shape:", daily_df.shape)


Processing C:\Users\Shriyans S Sahoo\IIIT Dharwad\Academics\Sem 4\Time Series Forecasting\TimeSeries Project\nldc_xls\2023-24

Processing C:\Users\Shriyans S Sahoo\IIIT Dharwad\Academics\Sem 4\Time Series Forecasting\TimeSeries Project\nldc_xls\2024-25

Processing C:\Users\Shriyans S Sahoo\IIIT Dharwad\Academics\Sem 4\Time Series Forecasting\TimeSeries Project\nldc_xls\2025-26
Daily dataset shape: (14, 2)


In [6]:
daily_df["date"] = pd.to_datetime(daily_df["date"])

daily_df["day_of_week"] = daily_df["date"].dt.dayofweek
daily_df["month"] = daily_df["date"].dt.month
daily_df["is_weekend"] = daily_df["day_of_week"].isin([5, 6]).astype(int)

In [7]:
HOLIDAYS_PATH = BASE_DIR / "holidays.csv"  # adjust if needed

holidays_df = pd.read_csv(HOLIDAYS_PATH)

holidays_df["date"] = pd.to_datetime(holidays_df["date"], dayfirst=True)

In [8]:
# Normalize columns
holidays_df.columns = holidays_df.columns.str.strip().str.lower()

# Convert date
holidays_df["date"] = pd.to_datetime(holidays_df["date"])

# Clean values
holidays_df["pre_post_flag"] = holidays_df["pre_post_flag"].astype(str).str.strip().str.lower()

# Filter only all-India holidays
holidays_df = holidays_df[holidays_df["impact_scope"] == "all_india"]

# Create features
holidays_df["is_festival"] = holidays_df["is_holiday"]
holidays_df["is_pre"] = (holidays_df["pre_post_flag"] == "pre").astype(int)
holidays_df["is_post"] = (holidays_df["pre_post_flag"] == "post").astype(int)

In [9]:
# Assign priority: festival > pre > post
def assign_priority(row):
    if row["is_festival"] == 1:
        return 3
    elif row["is_pre"] == 1:
        return 2
    elif row["is_post"] == 1:
        return 1
    else:
        return 0

holidays_df["priority"] = holidays_df.apply(assign_priority, axis=1)

# Sort and keep highest priority per date
holidays_df = holidays_df.sort_values(["date", "priority"], ascending=[True, False])

holidays_df = holidays_df.drop_duplicates(subset="date", keep="first")

# Keep only required columns
holidays_df = holidays_df[[
    "date",
    "is_festival",
    "is_pre",
    "is_post",
    "reference_festival"
]].rename(columns={"reference_festival": "festival_name"})

In [10]:
daily_df = daily_df.merge(holidays_df, on="date", how="left")

# Fill missing values
daily_df["is_festival"] = daily_df["is_festival"].fillna(0)
daily_df["is_pre"] = daily_df["is_pre"].fillna(0)
daily_df["is_post"] = daily_df["is_post"].fillna(0)
daily_df["festival_name"] = daily_df["festival_name"].fillna("none")

In [11]:
daily_df = daily_df.sort_values("date").reset_index(drop=True)

daily_path = OUTPUT_DIR / "daily_all_india_2023_26.csv"
daily_df.to_csv(daily_path, index=False)

print("Saved daily dataset:", daily_path)
print("Final shape:", daily_df.shape)

Saved daily dataset: C:\Users\Shriyans S Sahoo\IIIT Dharwad\Academics\Sem 4\Time Series Forecasting\TimeSeries Project\data\processed\daily_all_india_2023_26.csv
Final shape: (14, 9)


## Part B: 15-Min Data (2025-26)

In [12]:
def read_timeseries_sheet(file_path):
    try:
        xl = pd.read_excel(file_path, sheet_name=None, engine="xlrd")
    except:
        try:
            xl = pd.read_excel(file_path, sheet_name=None, engine="openpyxl")
        except:
            return None

    for sheet in xl.keys():
        if "time" in sheet.lower():
            return xl[sheet]

    return None

In [13]:
def parse_timeseries(file_path):
    df = read_timeseries_sheet(file_path)
    if df is None:
        return None

    date = extract_date_from_filename(file_path)
    if date is None:
        return None

    df.columns = df.columns.str.strip().str.lower()

    # Find time column
    time_cols = [col for col in df.columns if "time" in col]
    if len(time_cols) == 0:
        return None

    time_col = time_cols[0]

    # Create datetime
    df["datetime"] = pd.to_datetime(
        date.strftime("%Y-%m-%d") + " " + df[time_col].astype(str),
        errors="coerce"
    )

    df["date"] = df["datetime"].dt.date

    # Rename demand column
    for col in df.columns:
        if "demand" in col and "met" in col:
            df.rename(columns={col: "demand_met_mw"}, inplace=True)

    return df

In [19]:
def parse_timeseries(file_path):
    try:
        # Read raw sheet (no header assumption)
        df = pd.read_excel(file_path, sheet_name="TimeSeries", header=None)

        header_row = None

        # Step 1: Find header row dynamically
        for i in range(len(df)):
            row = df.iloc[i].astype(str).str.lower().tolist()

            if any(("time" in x) or ("block" in x) for x in row):
                header_row = i
                break

        if header_row is None:
            return None

        # Step 2: Extract header
        headers = df.iloc[header_row].astype(str).str.strip().tolist()

        # Step 3: Extract data below header
        data = df.iloc[header_row + 1:].copy()
        data.columns = headers

        # Step 4: Drop completely empty rows
        data = data.dropna(how="all")

        # Step 5: Identify columns
        time_col = data.columns[0]

        if len(data.columns) < 3:
            return None

        demand_col = data.columns[2]  # Based on NLDC structure

        # Step 6: Get date
        date = extract_date_from_filename(file_path)
        if date is None:
            return None

        # Step 7: Create datetime
        data["datetime"] = pd.to_datetime(
            date.strftime("%Y-%m-%d") + " " + data[time_col].astype(str),
            errors="coerce"
        )

        # Step 8: Extract demand
        data["demand_met_mw"] = pd.to_numeric(data[demand_col], errors="coerce")

        # Step 9: Keep only required columns
        data = data[["datetime", "demand_met_mw"]]

        return data

    except:
        return None

In [20]:
timeseries_list = []

folder_2025 = XLS_DIR / "2025-26"

files = list(folder_2025.rglob("*.xls*"))
print("Files found:", len(files))

for file_path in files:
    ts = parse_timeseries(file_path)
    if ts is not None:
        timeseries_list.append(ts)

timeseries_df = pd.concat(timeseries_list, ignore_index=True)

print("Timeseries shape:", timeseries_df.shape)

Files found: 349
Timeseries shape: (34101, 2)


In [21]:
# Remove invalid timestamps
timeseries_df = timeseries_df.dropna(subset=["datetime"])

# Sort properly
timeseries_df = timeseries_df.sort_values("datetime")

# Remove duplicates
timeseries_df = timeseries_df.drop_duplicates(subset=["datetime"])

# Reset index
timeseries_df = timeseries_df.reset_index(drop=True)

print("After cleaning:", timeseries_df.shape)

After cleaning: (33403, 2)


In [22]:
timeseries_df["date"] = timeseries_df["datetime"].dt.date
timeseries_df["hour"] = timeseries_df["datetime"].dt.hour
timeseries_df["minute"] = timeseries_df["datetime"].dt.minute
timeseries_df["day_of_week"] = timeseries_df["datetime"].dt.dayofweek

print("Feature columns added")

Feature columns added


In [23]:
print("\nHead:")
print(timeseries_df.head())

print("\nNull values:")
print(timeseries_df.isnull().sum())

print("\nDate range:")
print(timeseries_df["datetime"].min(), "→", timeseries_df["datetime"].max())


Head:
             datetime  demand_met_mw        date  hour  minute  day_of_week
0 2025-04-01 00:00:00       191368.0  2025-04-01     0       0            1
1 2025-04-01 00:15:00       190921.0  2025-04-01     0      15            1
2 2025-04-01 00:30:00       190222.0  2025-04-01     0      30            1
3 2025-04-01 00:45:00       188971.0  2025-04-01     0      45            1
4 2025-04-01 01:00:00       188162.0  2025-04-01     1       0            1

Null values:
datetime         0
demand_met_mw    0
date             0
hour             0
minute           0
day_of_week      0
dtype: int64

Date range:
2025-04-01 00:00:00 → 2026-03-31 23:45:00


In [24]:
ts_path = OUTPUT_DIR / "timeseries_15min_2025_26.csv"

timeseries_df.to_csv(ts_path, index=False)

print("Saved timeseries dataset at:", ts_path)

Saved timeseries dataset at: C:\Users\Shriyans S Sahoo\IIIT Dharwad\Academics\Sem 4\Time Series Forecasting\TimeSeries Project\data\processed\timeseries_15min_2025_26.csv
